# SQP & Interior Point Methods
## Nonlinear Constrained Optimization for Trajectory Planning

In [ ]:
%matplotlib inline

import numpy as np
from numpy.linalg import norm, solve, inv, det, eigvals
from scipy.optimize import minimize, linprog
from scipy.linalg import block_diag, cho_factor, cho_solve
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})

print("All imports successful.")

In [ ]:
# =============================================================================
# Constants
# =============================================================================

np.random.seed(42)

# Colors
PRIMARY = 'steelblue'
SECONDARY = 'coral'
TERTIARY = 'seagreen'
ACCENT = 'goldenrod'
EVOLUTION_CMAP = 'viridis'

# Tolerances
KKT_TOL = 1e-6
CONSTRAINT_TOL = 1e-8
MAX_ITER = 200
BARRIER_MU_INIT = 10.0
BARRIER_MU_FACTOR = 0.2
BARRIER_MU_TOL = 1e-8

# Cart-pole parameters
CART_MASS = 1.0
POLE_MASS = 0.3
POLE_LENGTH = 0.5
GRAVITY = 9.81
DT = 0.05
N_STEPS = 40

# Robot arm parameters
ARM_LINK_LENGTHS = np.array([1.0, 0.8, 0.6])
ARM_JOINT_LIMITS = np.array([[-np.pi, np.pi], [-np.pi/2, np.pi/2], [-np.pi/2, np.pi/2]])
ARM_TORQUE_LIMIT = 5.0

print("Constants defined.")

# =============================================================================
# Section 1 — Introduction
# =============================================================================

We study two foundational families of algorithms for solving **nonlinear constrained optimization** problems of the form

$$
\boxed{\min_{x} \; f(x) \quad \text{s.t.} \quad c_E(x) = 0, \quad c_I(x) \le 0}
$$

where $f : \mathbb{R}^n \to \mathbb{R}$, $c_E : \mathbb{R}^n \to \mathbb{R}^{m_E}$, and $c_I : \mathbb{R}^n \to \mathbb{R}^{m_I}$.

**Two algorithmic paradigms:**

| Method | Idea | Inequality handling |
|--------|------|---------------------|
| **Sequential Quadratic Programming (SQP)** | Solve a QP subproblem at each iteration (Newton on the KKT system) | Active-set strategy or L1 merit |
| **Interior Point / Barrier** | Replace inequalities with $-\mu \sum \log(-c_i(x))$ barrier | Central path $\mu \to 0$ |

**Prerequisites:** convex optimization, Newton's method, Lagrangian duality, KKT conditions.

**Primary reference:** Nocedal & Wright, *Numerical Optimization*, 2nd ed., Springer, 2006 — Chapters 12, 15, 18, 19.

# =============================================================================
# Section 2 — Equality-Constrained Newton's Method
# =============================================================================

Consider the equality-constrained problem:

$$\min_x \; f(x) \quad \text{s.t.} \quad c(x) = 0$$

The **Lagrangian** is $\mathcal{L}(x, \lambda) = f(x) + \lambda^T c(x)$.

The **KKT conditions** are:

$$\nabla_x \mathcal{L} = \nabla f(x) + J(x)^T \lambda = 0, \qquad c(x) = 0$$

Applying Newton's method to the KKT system yields the **saddle-point system**:

$$
\boxed{
\begin{pmatrix} W & J^T \\ J & 0 \end{pmatrix}
\begin{pmatrix} \Delta x \\ \Delta \lambda \end{pmatrix}
= -\begin{pmatrix} \nabla f + J^T \lambda \\ c \end{pmatrix}
}
$$

where $W = \nabla^2_{xx} \mathcal{L}$ is the Hessian of the Lagrangian and $J = \nabla c(x)^T$ is the constraint Jacobian.

In [ ]:
def equality_constrained_newton(f, grad_f, hess_f, c, jac_c, x0, lam0,
                                 max_iter=50, tol=1e-10):
    """Newton's method on the KKT system for equality-constrained optimization.

    Solves: min f(x)  s.t. c(x) = 0  via Newton steps on the KKT conditions.

    Args:
        f: Objective function f(x) -> float.
        grad_f: Gradient of f, grad_f(x) -> array. Shape: (n,).
        hess_f: Hessian of f, hess_f(x) -> array. Shape: (n, n).
        c: Equality constraints c(x) -> array. Shape: (m,).
        jac_c: Jacobian of c, jac_c(x) -> array. Shape: (m, n).
        x0: Initial primal variable. Shape: (n,).
        lam0: Initial dual variable. Shape: (m,).
        max_iter: Maximum number of iterations.
        tol: KKT residual tolerance.

    Returns:
        x_hist: List of primal iterates.
        lam_hist: List of dual iterates.
        kkt_residuals: List of KKT residual norms.
    """
    x = x0.copy().astype(float)
    lam = lam0.copy().astype(float)
    n = len(x)
    m = len(lam)

    x_hist = [x.copy()]
    lam_hist = [lam.copy()]
    kkt_residuals = []

    for k in range(max_iter):
        gf = grad_f(x)
        J = jac_c(x)
        cv = c(x)
        W = hess_f(x)

        # KKT residual
        r1 = gf + J.T @ lam
        r2 = cv
        kkt_res = norm(np.concatenate([r1, r2]))
        kkt_residuals.append(kkt_res)

        if kkt_res < tol:
            break

        # Build and solve saddle-point system
        KKT_mat = np.block([
            [W, J.T],
            [J, np.zeros((m, m))]
        ])
        rhs = -np.concatenate([r1, r2])
        delta = solve(KKT_mat, rhs)

        dx = delta[:n]
        dlam = delta[n:]

        x = x + dx
        lam = lam + dlam

        x_hist.append(x.copy())
        lam_hist.append(lam.copy())

    return x_hist, lam_hist, kkt_residuals

print("equality_constrained_newton defined.")

In [ ]:
# Test: Equality-constrained QP
# min 0.5 * x^T Q x + p^T x   s.t.  A x = b

Q = np.array([[2.0, 0.5], [0.5, 1.0]])
p_vec = np.array([-2.0, -1.0])
A_eq = np.array([[1.0, 1.0]])
b_eq = np.array([1.0])

def f_qp(x):
    return 0.5 * x @ Q @ x + p_vec @ x

def grad_f_qp(x):
    return Q @ x + p_vec

def hess_f_qp(x):
    return Q.copy()

def c_qp(x):
    return A_eq @ x - b_eq

def jac_c_qp(x):
    return A_eq.copy()

x0 = np.array([0.0, 0.0])
lam0 = np.array([0.0])

x_hist, lam_hist, kkt_res = equality_constrained_newton(
    f_qp, grad_f_qp, hess_f_qp, c_qp, jac_c_qp, x0, lam0)

x_sol = x_hist[-1]
lam_sol = lam_hist[-1]

print(f"Solution: x* = [{x_sol[0]:.6f}, {x_sol[1]:.6f}]")
print(f"Multiplier: lambda* = {lam_sol[0]:.6f}")
print(f"Constraint: c(x*) = {c_qp(x_sol)[0]:.2e}")
print(f"Converged in {len(kkt_res)} iterations")

constraint_err = abs(c_qp(x_sol)[0])
status = "PASS" if constraint_err < KKT_TOL else "FAIL"
print(f"\nequality_newton: constraint satisfaction = {constraint_err:.2e} [{status}]")

# =============================================================================
# Section 3 — Sequential Quadratic Programming (Theory)
# =============================================================================

For the general problem with both equality and inequality constraints, SQP solves a **QP subproblem** at each iterate $x_k$:

$$
\boxed{
\min_{d} \; \nabla f_k^T d + \tfrac{1}{2} d^T W_k d \quad \text{s.t.} \quad
c_E(x_k) + J_E(x_k) d = 0, \quad c_I(x_k) + J_I(x_k) d \le 0
}
$$

This is a **quadratic model** of the Lagrangian with **linearized constraints** — exactly a Newton step on the KKT system of the original NLP.

### L1 Merit Function

To ensure global convergence, we use the **exact L1 penalty / merit function**:

$$
\phi_1(x; \rho) = f(x) + \rho \left( \| c_E(x) \|_1 + \| \max(0, c_I(x)) \|_1 \right)
$$

The penalty parameter $\rho$ must satisfy $\rho > \|\lambda^*\|_\infty$ for the merit function to be exact.

We accept a step $\alpha$ via **Armijo** condition on $\phi_1$:

$$
\phi_1(x_k + \alpha d_k; \rho) \le \phi_1(x_k; \rho) + \eta \, \alpha \, D\phi_1(x_k; d_k, \rho)
$$

# =============================================================================
# Section 4 — SQP Implementation
# =============================================================================

In [ ]:
def merit_function_l1(x, f, c_eq, c_ineq, rho):
    """Compute the L1 exact penalty merit function.

    Args:
        x: Current point. Shape: (n,).
        f: Objective function f(x) -> float.
        c_eq: Equality constraint function c_eq(x) -> array. Shape: (m_e,).
        c_ineq: Inequality constraint function c_ineq(x) -> array. Shape: (m_i,).
            Convention: c_ineq(x) <= 0.
        rho: Penalty parameter (positive scalar).

    Returns:
        phi: Value of the L1 merit function (float).
    """
    fval = f(x)
    eq_viol = norm(c_eq(x), 1) if c_eq is not None else 0.0
    ineq_vals = c_ineq(x) if c_ineq is not None else np.array([])
    ineq_viol = np.sum(np.maximum(0, ineq_vals)) if len(ineq_vals) > 0 else 0.0
    return fval + rho * (eq_viol + ineq_viol)


def solve_qp_subproblem(H, g, A_eq, b_eq, A_ineq, b_ineq):
    """Solve the QP subproblem using scipy.optimize.minimize with SLSQP.

    Solves: min 0.5 d^T H d + g^T d
            s.t. A_eq d + b_eq = 0
                 A_ineq d + b_ineq <= 0

    Args:
        H: Hessian matrix. Shape: (n, n).
        g: Gradient vector. Shape: (n,).
        A_eq: Equality constraint Jacobian. Shape: (m_e, n) or None.
        b_eq: Equality constraint values. Shape: (m_e,) or None.
        A_ineq: Inequality constraint Jacobian. Shape: (m_i, n) or None.
        b_ineq: Inequality constraint values. Shape: (m_i,) or None.

    Returns:
        d: Search direction. Shape: (n,).
        lam_eq: Equality multipliers. Shape: (m_e,).
        lam_ineq: Inequality multipliers. Shape: (m_i,).
    """
    n = len(g)
    constraints = []

    if A_eq is not None and len(A_eq) > 0:
        constraints.append({
            'type': 'eq',
            'fun': lambda d, A=A_eq, b=b_eq: A @ d + b,
            'jac': lambda d, A=A_eq: A
        })

    if A_ineq is not None and len(A_ineq) > 0:
        # scipy convention: c(d) >= 0, our convention: A d + b <= 0 => -(A d + b) >= 0
        constraints.append({
            'type': 'ineq',
            'fun': lambda d, A=A_ineq, b=b_ineq: -(A @ d + b),
            'jac': lambda d, A=A_ineq: -A
        })

    def qp_obj(d):
        return 0.5 * d @ H @ d + g @ d

    def qp_grad(d):
        return H @ d + g

    d0 = np.zeros(n)
    res = minimize(qp_obj, d0, jac=qp_grad, constraints=constraints, method='SLSQP',
                   options={'ftol': 1e-12, 'maxiter': 500, 'disp': False})
    d = res.x

    # Estimate multipliers from KKT of the QP
    grad_at_d = H @ d + g
    m_e = len(A_eq) if A_eq is not None else 0
    m_i = len(A_ineq) if A_ineq is not None else 0
    lam_eq = np.zeros(m_e)
    lam_ineq = np.zeros(m_i)

    if m_e + m_i > 0:
        # Solve least-squares:  [A_eq^T, A_ineq^T] [lam_eq; lam_ineq] = -grad_at_d
        parts = []
        if m_e > 0:
            parts.append(A_eq.T)
        if m_i > 0:
            parts.append(A_ineq.T)
        Jt = np.hstack(parts)
        lam_all, _, _, _ = np.linalg.lstsq(Jt, -grad_at_d, rcond=None)
        if m_e > 0:
            lam_eq = lam_all[:m_e]
        if m_i > 0:
            lam_ineq = lam_all[m_e:]
            lam_ineq = np.maximum(lam_ineq, 0)

    return d, lam_eq, lam_ineq


def sqp_solve(f, grad_f, hess_f, c_eq, jac_eq, c_ineq, jac_ineq, x0,
              max_iter=MAX_ITER, tol=KKT_TOL, rho_init=10.0):
    """Full SQP solver with L1 merit function line search.

    Solves: min f(x)  s.t. c_eq(x)=0, c_ineq(x)<=0.

    Args:
        f: Objective function.
        grad_f: Gradient of objective. Shape: (n,).
        hess_f: Hessian of Lagrangian (approx). Shape: (n, n).
        c_eq: Equality constraints c_eq(x) -> array or None.
        jac_eq: Jacobian of equality constraints or None.
        c_ineq: Inequality constraints c_ineq(x) -> array or None.
        jac_ineq: Jacobian of inequality constraints or None.
        x0: Initial point. Shape: (n,).
        max_iter: Maximum iterations.
        tol: KKT tolerance.
        rho_init: Initial merit function penalty.

    Returns:
        result: Dict with keys 'x', 'fun', 'lam_eq', 'lam_ineq',
                'x_hist', 'kkt_hist', 'merit_hist', 'converged'.
    """
    x = x0.copy().astype(float)
    n = len(x)
    rho = rho_init

    # Initialize multipliers
    m_e = len(c_eq(x)) if c_eq is not None else 0
    m_i = len(c_ineq(x)) if c_ineq is not None else 0
    lam_eq = np.zeros(m_e)
    lam_ineq = np.zeros(m_i)

    x_hist = [x.copy()]
    kkt_hist = []
    merit_hist = []
    converged = False

    for k in range(max_iter):
        gf = grad_f(x)
        W = hess_f(x, lam_eq, lam_ineq)

        # Regularize Hessian if needed
        eig_min = np.min(np.real(eigvals(W)))
        if eig_min < 1e-6:
            W = W + (1e-6 - eig_min) * np.eye(n)

        A_e = jac_eq(x) if jac_eq is not None else None
        b_e = c_eq(x) if c_eq is not None else None
        A_i = jac_ineq(x) if jac_ineq is not None else None
        b_i = c_ineq(x) if c_ineq is not None else None

        # Solve QP subproblem
        d, lam_eq_new, lam_ineq_new = solve_qp_subproblem(W, gf, A_e, b_e, A_i, b_i)

        # Update penalty parameter
        lam_max = 0.0
        if m_e > 0:
            lam_max = max(lam_max, norm(lam_eq_new, np.inf))
        if m_i > 0:
            lam_max = max(lam_max, norm(lam_ineq_new, np.inf))
        rho = max(rho, 1.1 * lam_max + 0.1)

        # KKT residual
        kkt_r = gf.copy()
        if m_e > 0:
            kkt_r += A_e.T @ lam_eq_new
        if m_i > 0:
            kkt_r += A_i.T @ lam_ineq_new
        eq_viol = norm(b_e) if b_e is not None else 0.0
        ineq_viol = norm(np.maximum(0, b_i)) if b_i is not None else 0.0
        kkt_norm = norm(kkt_r) + eq_viol + ineq_viol
        kkt_hist.append(kkt_norm)

        phi_current = merit_function_l1(x, f, c_eq, c_ineq, rho)
        merit_hist.append(phi_current)

        if kkt_norm < tol:
            converged = True
            lam_eq = lam_eq_new
            lam_ineq = lam_ineq_new
            break

        # Armijo line search on L1 merit
        alpha = 1.0
        eta = 1e-4
        for _ in range(30):
            x_trial = x + alpha * d
            phi_trial = merit_function_l1(x_trial, f, c_eq, c_ineq, rho)
            # Directional derivative approximation
            dphi = phi_trial - phi_current
            if phi_trial <= phi_current - eta * alpha * (norm(d)**2):
                break
            alpha *= 0.5

        x = x + alpha * d
        lam_eq = lam_eq_new
        lam_ineq = lam_ineq_new

        x_hist.append(x.copy())

    return {
        'x': x.copy(),
        'fun': f(x),
        'lam_eq': lam_eq,
        'lam_ineq': lam_ineq,
        'x_hist': x_hist,
        'kkt_hist': kkt_hist,
        'merit_hist': merit_hist,
        'converged': converged,
        'iterations': len(kkt_hist)
    }

print("sqp_solve, merit_function_l1, solve_qp_subproblem defined.")

In [ ]:
# Test SQP on Rosenbrock with equality constraint
# min (1-x1)^2 + 100*(x2-x1^2)^2   s.t. x1 + x2 = 1

def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

def grad_rosenbrock(x):
    g = np.zeros(2)
    g[0] = -2*(1 - x[0]) + 200*(x[1] - x[0]**2)*(-2*x[0])
    g[1] = 200*(x[1] - x[0]**2)
    return g

def hess_rosenbrock(x, lam_eq=None, lam_ineq=None):
    H = np.zeros((2, 2))
    H[0, 0] = 2 + 200*(3*x[0]**2 - x[1])*2 + 400*x[0]**2
    H[0, 0] = 2 - 400*(x[1] - 3*x[0]**2)
    H[0, 1] = -400*x[0]
    H[1, 0] = -400*x[0]
    H[1, 1] = 200
    return H

def c_eq_rosen(x):
    return np.array([x[0] + x[1] - 1.0])

def jac_eq_rosen(x):
    return np.array([[1.0, 1.0]])

x0_rosen = np.array([-1.0, 2.0])

result_sqp = sqp_solve(
    rosenbrock, grad_rosenbrock, hess_rosenbrock,
    c_eq_rosen, jac_eq_rosen, None, None,
    x0_rosen, max_iter=100, tol=KKT_TOL
)

x_star = result_sqp['x']
print(f"SQP solution: x* = [{x_star[0]:.8f}, {x_star[1]:.8f}]")
print(f"Objective: f(x*) = {result_sqp['fun']:.8e}")
print(f"Constraint: c(x*) = {c_eq_rosen(x_star)[0]:.2e}")
print(f"Iterations: {result_sqp['iterations']}")
print(f"Converged: {result_sqp['converged']}")

kkt_final = result_sqp['kkt_hist'][-1] if result_sqp['kkt_hist'] else float('inf')
status = "PASS" if kkt_final < KKT_TOL else "FAIL"
print(f"\nsqp_rosenbrock_eq: KKT residual = {kkt_final:.2e} [{status}]")

In [ ]:
# Test SQP with inequality constraints
# min (x1-1)^2 + (x2-2.5)^2
# s.t. x1 - 2*x2 + 2 >= 0    =>  -(x1 - 2*x2 + 2) <= 0
#      -x1 - 2*x2 + 6 >= 0   =>  -(-x1 - 2*x2 + 6) <= 0
#      -x1 + 2*x2 + 2 >= 0   =>  -(-x1 + 2*x2 + 2) <= 0

def f_ineq(x):
    return (x[0] - 1)**2 + (x[1] - 2.5)**2

def grad_f_ineq(x):
    return np.array([2*(x[0] - 1), 2*(x[1] - 2.5)])

def hess_f_ineq(x, lam_eq=None, lam_ineq=None):
    return np.array([[2.0, 0.0], [0.0, 2.0]])

def c_ineq_test(x):
    return np.array([
        -(x[0] - 2*x[1] + 2),
        -(-x[0] - 2*x[1] + 6),
        -(-x[0] + 2*x[1] + 2),
    ])

def jac_ineq_test(x):
    return np.array([
        [-1.0, 2.0],
        [1.0, 2.0],
        [1.0, -2.0],
    ])

x0_ineq = np.array([2.0, 0.0])

result_sqp_ineq = sqp_solve(
    f_ineq, grad_f_ineq, hess_f_ineq,
    None, None, c_ineq_test, jac_ineq_test,
    x0_ineq, max_iter=100, tol=KKT_TOL
)

x_star_ineq = result_sqp_ineq['x']
print(f"SQP (ineq) solution: x* = [{x_star_ineq[0]:.6f}, {x_star_ineq[1]:.6f}]")
print(f"Objective: f(x*) = {result_sqp_ineq['fun']:.6f}")
print(f"Constraint values: {c_ineq_test(x_star_ineq)}")
print(f"Iterations: {result_sqp_ineq['iterations']}")

c_sat = np.all(c_ineq_test(x_star_ineq) <= CONSTRAINT_TOL)
status = "PASS" if c_sat else "FAIL"
print(f"\nsqp_ineq: constraint satisfaction = {np.max(c_ineq_test(x_star_ineq)):.2e} [{status}]")

# =============================================================================
# Section 5 — Interior Point / Barrier Methods
# =============================================================================

For inequality-constrained problems, the **log-barrier** approach replaces

$$\min_x f(x) \quad \text{s.t.} \quad c_i(x) \le 0, \; i=1,\ldots,m$$

with the **barrier subproblem**:

$$\boxed{\min_x \; f(x) - \mu \sum_{i=1}^{m} \log(-c_i(x))}$$

As $\mu \to 0$, the solution of the barrier subproblem converges to the constrained optimum. The set of minimizers parametrized by $\mu$ is the **central path**.

### Phase I: Finding a Feasible Point

Before the barrier method can begin, we need a strictly feasible point ($c_i(x) < 0$ for all $i$). A **Phase I** approach solves:

$$\min_s \; s \quad \text{s.t.} \quad c_i(x) \le s, \; \forall i$$

starting from $s > \max_i c_i(x_0)$. If $s^* < 0$, the original problem is feasible.

In [ ]:
def barrier_method(f, grad_f, hess_f, c_ineq, jac_ineq, x0,
                   mu_init=BARRIER_MU_INIT, mu_factor=BARRIER_MU_FACTOR,
                   mu_tol=BARRIER_MU_TOL, max_outer=30, max_inner=50,
                   tol_inner=1e-8):
    """Log-barrier method for inequality-constrained optimization.

    Solves: min f(x)  s.t. c_ineq(x) <= 0.

    Args:
        f: Objective function.
        grad_f: Gradient of objective. Shape: (n,).
        hess_f: Hessian of objective. Shape: (n, n).
        c_ineq: Inequality constraints c(x) -> array. Shape: (m,).
        jac_ineq: Jacobian of constraints. Shape: (m, n).
        x0: Initial strictly feasible point. Shape: (n,).
        mu_init: Initial barrier parameter.
        mu_factor: Reduction factor for mu.
        mu_tol: Stopping tolerance on mu.
        max_outer: Maximum outer iterations.
        max_inner: Maximum inner Newton iterations.
        tol_inner: Inner Newton tolerance.

    Returns:
        result: Dict with 'x', 'fun', 'x_hist', 'mu_hist', 'converged'.
    """
    x = x0.copy().astype(float)
    n = len(x)
    mu = mu_init

    x_hist = [x.copy()]
    mu_hist = [mu]
    outer_converged = False

    for outer in range(max_outer):
        # Inner: minimize f(x) - mu * sum(log(-c_i(x))) via Newton
        for inner in range(max_inner):
            cv = c_ineq(x)
            # Check feasibility
            if np.any(cv >= 0):
                # Project back slightly
                x = x - 0.01 * jac_ineq(x).T @ np.maximum(cv, 0)
                cv = c_ineq(x)
                if np.any(cv >= 0):
                    break

            J = jac_ineq(x)
            inv_c = 1.0 / (-cv)  # Shape: (m,)

            # Barrier gradient: grad_f - mu * sum( (1/(-c_i)) * grad_c_i )
            barrier_grad = grad_f(x) + mu * J.T @ inv_c

            # Barrier Hessian: hess_f + mu * sum( (1/c_i^2) * grad_c_i grad_c_i^T )
            S = np.diag(inv_c**2)
            barrier_hess = hess_f(x) + mu * J.T @ S @ J

            # Regularize
            eig_min = np.min(np.real(eigvals(barrier_hess)))
            if eig_min < 1e-8:
                barrier_hess += (1e-8 - eig_min) * np.eye(n)

            # Newton step
            try:
                dx = solve(barrier_hess, -barrier_grad)
            except np.linalg.LinAlgError:
                break

            # Backtracking to maintain feasibility
            alpha = 1.0
            for _ in range(40):
                x_trial = x + alpha * dx
                cv_trial = c_ineq(x_trial)
                if np.all(cv_trial < 0):
                    break
                alpha *= 0.5
            else:
                break

            # Armijo on barrier objective
            barrier_obj = lambda z: f(z) - mu * np.sum(np.log(-c_ineq(z)))
            phi0 = barrier_obj(x)
            for _ in range(20):
                x_trial = x + alpha * dx
                if barrier_obj(x_trial) <= phi0 - 1e-4 * alpha * barrier_grad @ dx:
                    break
                alpha *= 0.5

            x = x + alpha * dx

            if norm(alpha * dx) < tol_inner:
                break

        x_hist.append(x.copy())
        mu *= mu_factor
        mu_hist.append(mu)

        if mu < mu_tol:
            outer_converged = True
            break

    return {
        'x': x.copy(),
        'fun': f(x),
        'x_hist': x_hist,
        'mu_hist': mu_hist,
        'converged': outer_converged
    }

print("barrier_method defined.")

In [ ]:
# Test barrier method on the same inequality-constrained problem
# Need a strictly feasible starting point
x0_barrier = np.array([0.5, 1.0])  # strictly feasible
cv0 = c_ineq_test(x0_barrier)
print(f"Initial constraint values: {cv0} (all < 0? {np.all(cv0 < 0)})")

result_barrier = barrier_method(
    f_ineq, grad_f_ineq,
    lambda x: np.array([[2.0, 0.0], [0.0, 2.0]]),
    c_ineq_test, jac_ineq_test,
    x0_barrier, mu_init=10.0, mu_factor=0.2, mu_tol=1e-10
)

x_bar = result_barrier['x']
print(f"\nBarrier solution: x* = [{x_bar[0]:.6f}, {x_bar[1]:.6f}]")
print(f"Objective: f(x*) = {result_barrier['fun']:.6f}")
print(f"Constraint values: {c_ineq_test(x_bar)}")

mu_final = result_barrier['mu_hist'][-1]
status = "PASS" if mu_final < 1e-6 else "FAIL"
print(f"\nbarrier: mu -> 0 check, mu_final = {mu_final:.2e} [{status}]")

c_sat = np.all(c_ineq_test(x_bar) <= CONSTRAINT_TOL)
status2 = "PASS" if c_sat else "FAIL"
print(f"barrier: constraint satisfaction = {np.max(c_ineq_test(x_bar)):.2e} [{status2}]")

# =============================================================================
# Section 6 — Primal-Dual Interior Point Method
# =============================================================================

The **primal-dual** interior point method directly solves the perturbed KKT system. Introducing slack variables $s \ge 0$ and multipliers $\lambda \ge 0$:

$$c_I(x) + s = 0, \quad S \Lambda e = \mu e$$

where $S = \text{diag}(s)$, $\Lambda = \text{diag}(\lambda)$, and $e$ is the ones vector.

The **complementarity** condition $s_i \lambda_i = \mu$ is relaxed to approach zero.

### Mehrotra Predictor-Corrector

1. **Predictor (affine) step**: Solve KKT with $\mu = 0$ to get $(\Delta x^{\text{aff}}, \Delta s^{\text{aff}}, \Delta \lambda^{\text{aff}})$
2. **Centering parameter**: $\sigma = \left(\frac{\mu^{\text{aff}}}{\mu}\right)^3$
3. **Corrector step**: Solve with $\mu_{\text{target}} = \sigma \mu$ and second-order correction

In [ ]:
def primal_dual_ip(f, grad_f, hess_f, c_ineq, jac_ineq, x0,
                   mu_init=1.0, max_iter=100, tol=KKT_TOL):
    """Primal-dual interior point method with Mehrotra predictor-corrector.

    Solves: min f(x)  s.t. c_ineq(x) <= 0.

    Args:
        f: Objective function.
        grad_f: Gradient. Shape: (n,).
        hess_f: Hessian. Shape: (n, n).
        c_ineq: Inequality constraints. Shape: (m,).
        jac_ineq: Jacobian. Shape: (m, n).
        x0: Initial point (need not be feasible). Shape: (n,).
        mu_init: Initial complementarity parameter.
        max_iter: Maximum iterations.
        tol: KKT tolerance.

    Returns:
        result: Dict with 'x', 'fun', 's', 'lam', 'x_hist', 'mu_hist', 'converged'.
    """
    x = x0.copy().astype(float)
    n = len(x)
    m = len(c_ineq(x))

    # Initialize slacks and multipliers
    cv = c_ineq(x)
    s = np.maximum(-cv, 0.1 * np.ones(m))
    lam = mu_init / s

    x_hist = [x.copy()]
    mu_hist = []
    converged = False

    for k in range(max_iter):
        cv = c_ineq(x)
        J = jac_ineq(x)
        gf = grad_f(x)
        H = hess_f(x)

        # Current mu
        mu = np.dot(s, lam) / m
        mu_hist.append(mu)

        # KKT residuals
        r_dual = gf + J.T @ lam
        r_primal = cv + s
        r_comp = s * lam - mu * np.ones(m)

        kkt_res = norm(np.concatenate([r_dual, r_primal, r_comp]))
        if kkt_res < tol and mu < tol:
            converged = True
            break

        # Regularize Hessian
        eig_min = np.min(np.real(eigvals(H)))
        if eig_min < 1e-8:
            H = H + (1e-8 - eig_min) * np.eye(n)

        S = np.diag(s)
        Lam = np.diag(lam)
        S_inv = np.diag(1.0 / s)

        # --- Predictor (affine) step: solve with mu=0 ---
        r_comp_aff = s * lam  # mu=0

        # Eliminate ds: ds = -s - S Lam^{-1} (r_comp_aff) + ... -> use Schur complement
        # Reduced system: (H + J^T Lam S^{-1} J) dx = -(r_dual - J^T Lam S^{-1} r_primal + J^T S^{-1} r_comp_aff)
        Sigma = Lam @ S_inv
        M = H + J.T @ Sigma @ J

        rhs_x = -(r_dual - J.T @ (Sigma @ r_primal) + J.T @ (S_inv @ r_comp_aff))

        try:
            dx_aff = solve(M, rhs_x)
        except np.linalg.LinAlgError:
            break

        ds_aff = -r_primal - J @ dx_aff
        dlam_aff = -S_inv @ (r_comp_aff + Lam @ ds_aff)

        # Step size for affine step (maintain s>0, lam>0)
        alpha_s_aff = 1.0
        alpha_l_aff = 1.0
        for i in range(m):
            if ds_aff[i] < 0:
                alpha_s_aff = min(alpha_s_aff, -0.995 * s[i] / ds_aff[i])
            if dlam_aff[i] < 0:
                alpha_l_aff = min(alpha_l_aff, -0.995 * lam[i] / dlam_aff[i])

        # Affine complementarity
        mu_aff = np.dot(s + alpha_s_aff * ds_aff, lam + alpha_l_aff * dlam_aff) / m

        # Centering parameter (Mehrotra)
        sigma = (mu_aff / mu) ** 3 if mu > 1e-20 else 0.0
        sigma = min(sigma, 1.0)

        # --- Corrector step ---
        r_comp_cc = s * lam - sigma * mu * np.ones(m) + ds_aff * dlam_aff

        rhs_x_cc = -(r_dual - J.T @ (Sigma @ r_primal) + J.T @ (S_inv @ r_comp_cc))

        try:
            dx = solve(M, rhs_x_cc)
        except np.linalg.LinAlgError:
            break

        ds = -r_primal - J @ dx
        dlam = -S_inv @ (r_comp_cc + Lam @ ds)

        # Step size
        alpha_s = 1.0
        alpha_l = 1.0
        for i in range(m):
            if ds[i] < 0:
                alpha_s = min(alpha_s, -0.995 * s[i] / ds[i])
            if dlam[i] < 0:
                alpha_l = min(alpha_l, -0.995 * lam[i] / dlam[i])

        alpha = min(alpha_s, alpha_l)

        x = x + alpha * dx
        s = s + alpha * ds
        lam = lam + alpha * dlam

        # Safety: ensure positivity
        s = np.maximum(s, 1e-14)
        lam = np.maximum(lam, 1e-14)

        x_hist.append(x.copy())

    return {
        'x': x.copy(),
        'fun': f(x),
        's': s.copy(),
        'lam': lam.copy(),
        'x_hist': x_hist,
        'mu_hist': mu_hist,
        'converged': converged,
        'iterations': len(mu_hist)
    }

print("primal_dual_ip defined.")

In [ ]:
# Test primal-dual IP on the inequality-constrained problem
x0_pdip = np.array([0.5, 1.0])

result_pdip = primal_dual_ip(
    f_ineq, grad_f_ineq,
    lambda x: np.array([[2.0, 0.0], [0.0, 2.0]]),
    c_ineq_test, jac_ineq_test,
    x0_pdip, mu_init=1.0, max_iter=100
)

x_pdip = result_pdip['x']
print(f"Primal-Dual IP solution: x* = [{x_pdip[0]:.6f}, {x_pdip[1]:.6f}]")
print(f"Objective: f(x*) = {result_pdip['fun']:.6f}")
print(f"Constraints: {c_ineq_test(x_pdip)}")
print(f"Slacks: {result_pdip['s']}")
print(f"Multipliers: {result_pdip['lam']}")
print(f"Iterations: {result_pdip['iterations']}")

mu_final_pdip = result_pdip['mu_hist'][-1] if result_pdip['mu_hist'] else float('inf')
status = "PASS" if mu_final_pdip < 1e-4 else "FAIL"
print(f"\npdip: mu -> 0 check, mu_final = {mu_final_pdip:.2e} [{status}]")

c_vals = c_ineq_test(x_pdip)
c_sat = np.all(c_vals <= CONSTRAINT_TOL)
status2 = "PASS" if c_sat else "FAIL"
print(f"pdip: constraint satisfaction = {np.max(c_vals):.2e} [{status2}]")

# =============================================================================
# Section 7 — Application: Cart-Pole Swing-Up Trajectory Optimization
# =============================================================================

We formulate the cart-pole swing-up as a **direct transcription** NLP. The state is $z = (x, \dot{x}, \theta, \dot{\theta})$ and control is $u$ (force on cart).

**Dynamics** (Euler discretization as equality constraints):

$$z_{k+1} = z_k + \Delta t \cdot f_{\text{dyn}}(z_k, u_k)$$

**Objective**: Minimize control effort $\sum u_k^2$ with terminal cost on final state.

**Constraints**:
- Equality: dynamics at each time step
- Inequality: control bounds $|u_k| \le u_{\max}$, track bounds $|x_k| \le x_{\max}$
- Terminal: $\theta_N = \pi$ (upright), $\dot{\theta}_N \approx 0$

In [ ]:
def cart_pole_dynamics(z, u, mc=CART_MASS, mp=POLE_MASS, l=POLE_LENGTH, g=GRAVITY):
    """Cart-pole continuous dynamics.

    Args:
        z: State [x, x_dot, theta, theta_dot]. Shape: (4,).
        u: Control force on cart (scalar).
        mc: Cart mass.
        mp: Pole mass.
        l: Pole half-length.
        g: Gravitational acceleration.

    Returns:
        z_dot: State derivative. Shape: (4,).
    """
    x, xd, th, thd = z
    s, c = np.sin(th), np.cos(th)
    total_m = mc + mp

    # Equations of motion
    denom = total_m - mp * c**2
    xdd = (u + mp * s * (l * thd**2 + g * c)) / denom
    thdd = (-u * c - mp * l * thd**2 * c * s - total_m * g * s) / (l * denom)

    return np.array([xd, xdd, thd, thdd])


def cart_pole_nlp(N=N_STEPS, dt=DT):
    """Set up and solve the cart-pole swing-up NLP using SQP via scipy.

    Uses direct transcription: decision variables are [z_0, u_0, z_1, u_1, ...].

    Args:
        N: Number of time steps.
        dt: Time step size.

    Returns:
        result: Dict with 'z_traj', 'u_traj', 'success', 'cost'.
    """
    nz = 4  # state dim
    nu = 1  # control dim
    nvar_per_step = nz + nu
    n_total = (N + 1) * nz + N * nu  # z_0..z_N, u_0..u_{N-1}

    # Indices
    def z_idx(k):
        return slice(k * nz, k * nz + nz)

    def u_idx(k):
        return (N + 1) * nz + k * nu

    # Initial and target states
    z_init = np.array([0.0, 0.0, 0.0, 0.0])  # hanging down (theta=0)
    z_target = np.array([0.0, 0.0, np.pi, 0.0])  # upright (theta=pi)

    U_MAX = 15.0
    X_MAX = 2.0

    # Objective: control effort + terminal cost
    Q_terminal = np.diag([1.0, 1.0, 100.0, 10.0])
    R = 0.01

    def objective(w):
        cost = 0.0
        for k in range(N):
            uk = w[u_idx(k)]
            cost += R * uk**2
        z_N = w[z_idx(N)]
        diff = z_N - z_target
        cost += diff @ Q_terminal @ diff
        return cost

    def grad_objective(w):
        g = np.zeros(n_total)
        for k in range(N):
            idx = u_idx(k)
            g[idx] = 2 * R * w[idx]
        z_N = w[z_idx(N)]
        diff = z_N - z_target
        g[z_idx(N)] = 2 * Q_terminal @ diff
        return g

    # Dynamics constraints: z_{k+1} - z_k - dt * f(z_k, u_k) = 0
    def dynamics_constraints(w):
        ceq = []
        for k in range(N):
            zk = w[z_idx(k)]
            uk = w[u_idx(k)]
            zk1 = w[z_idx(k + 1)]
            residual = zk1 - zk - dt * cart_pole_dynamics(zk, uk)
            ceq.extend(residual.tolist())
        # Initial condition
        ceq.extend((w[z_idx(0)] - z_init).tolist())
        return np.array(ceq)

    # Bounds
    bounds = []
    for k in range(N + 1):
        bounds.append((-X_MAX, X_MAX))   # x
        bounds.append((-10, 10))          # x_dot
        bounds.append((-2*np.pi, 2*np.pi))  # theta
        bounds.append((-15, 15))          # theta_dot
    for k in range(N):
        bounds.append((-U_MAX, U_MAX))    # u

    # Initial guess: linear interpolation
    w0 = np.zeros(n_total)
    for k in range(N + 1):
        alpha = k / N
        w0[z_idx(k)] = (1 - alpha) * z_init + alpha * z_target

    # Solve with SLSQP
    constraints = [{'type': 'eq', 'fun': dynamics_constraints}]

    res = minimize(objective, w0, jac=grad_objective, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'maxiter': 500, 'ftol': 1e-8, 'disp': False})

    # Extract trajectories
    z_traj = np.array([res.x[z_idx(k)] for k in range(N + 1)])
    u_traj = np.array([res.x[u_idx(k)] for k in range(N)])

    return {
        'z_traj': z_traj,
        'u_traj': u_traj,
        'success': res.success,
        'cost': res.fun,
        'message': res.message
    }

print("cart_pole_dynamics, cart_pole_nlp defined.")

In [ ]:
# Solve cart-pole swing-up
print("Solving cart-pole swing-up trajectory optimization...")
cp_result = cart_pole_nlp(N=N_STEPS, dt=DT)

print(f"Optimization success: {cp_result['success']}")
print(f"Message: {cp_result['message']}")
print(f"Cost: {cp_result['cost']:.4f}")

z_final = cp_result['z_traj'][-1]
print(f"\nFinal state: x={z_final[0]:.3f}, xd={z_final[1]:.3f}, "
      f"theta={z_final[2]:.3f} (target={np.pi:.3f}), thetad={z_final[3]:.3f}")

theta_err = abs(z_final[2] - np.pi)
status = "PASS" if theta_err < 0.3 else "FAIL"
print(f"\ncart_pole: theta error = {theta_err:.2e} [{status}]")

# =============================================================================
# Section 8 — Application: Constrained 3R Robot Arm Motion Planning
# =============================================================================

A planar 3R manipulator must move from an initial configuration to a target while respecting:
- **Joint limits**: $q_{\min} \le q \le q_{\max}$
- **Torque limits**: $|\tau_i| \le \tau_{\max}$
- **Obstacle avoidance**: Signed distance to circular obstacle > safety margin

In [ ]:
def forward_kinematics(q, link_lengths=ARM_LINK_LENGTHS):
    """Compute end-effector and all joint positions for a planar 3R arm.

    Args:
        q: Joint angles. Shape: (3,).
        link_lengths: Link lengths. Shape: (3,).

    Returns:
        positions: Joint positions including base and end-effector. Shape: (4, 2).
    """
    positions = [np.array([0.0, 0.0])]
    angle = 0.0
    for i in range(3):
        angle += q[i]
        pos = positions[-1] + link_lengths[i] * np.array([np.cos(angle), np.sin(angle)])
        positions.append(pos)
    return np.array(positions)


def robot_arm_nlp(q_start, q_goal, obstacle_center, obstacle_radius,
                  safety_margin=0.1, N_wp=10):
    """Solve constrained robot arm motion planning.

    Decision variables: waypoints q_1, ..., q_{N_wp} (q_0=start, q_{N+1}=goal fixed).

    Args:
        q_start: Starting joint config. Shape: (3,).
        q_goal: Goal joint config. Shape: (3,).
        obstacle_center: Center of circular obstacle. Shape: (2,).
        obstacle_radius: Radius of obstacle.
        safety_margin: Minimum clearance from obstacle.
        N_wp: Number of waypoints.

    Returns:
        result: Dict with 'q_traj', 'success', 'min_clearance'.
    """
    n_joints = 3
    n_vars = N_wp * n_joints

    def idx(k):
        return slice(k * n_joints, (k + 1) * n_joints)

    # Objective: minimize total joint displacement (smoothness)
    def objective(w):
        cost = 0.0
        q_prev = q_start
        for k in range(N_wp):
            q_k = w[idx(k)]
            cost += np.sum((q_k - q_prev)**2)
            q_prev = q_k
        cost += np.sum((q_goal - q_prev)**2)
        return cost

    def grad_objective(w):
        g = np.zeros(n_vars)
        q_prev = q_start
        for k in range(N_wp):
            q_k = w[idx(k)]
            g[idx(k)] += 2 * (q_k - q_prev)
            if k > 0:
                g[idx(k-1)] -= 2 * (q_k - w[idx(k-1)])
            q_prev = q_k
        # Last segment to goal
        g[idx(N_wp-1)] += 2 * (w[idx(N_wp-1)] - q_goal) * (-1)  # Correction
        g[idx(N_wp-1)] = 2 * (w[idx(N_wp-1)] - (q_goal + w[idx(N_wp-2)])/1) # Simplify
        # Recompute cleanly
        g = np.zeros(n_vars)
        q_all = [q_start] + [w[idx(k)] for k in range(N_wp)] + [q_goal]
        for k in range(N_wp):
            kk = k + 1  # index in q_all
            g[idx(k)] = 2 * (q_all[kk] - q_all[kk-1]) - 2 * (q_all[kk+1] - q_all[kk])
        return g

    # Obstacle avoidance: for each waypoint, check all link midpoints
    def obstacle_constraints(w):
        """Returns array where each entry should be >= 0 for feasibility."""
        clearances = []
        for k in range(N_wp):
            q_k = w[idx(k)]
            positions = forward_kinematics(q_k)
            for j in range(len(positions)):
                dist = norm(positions[j] - obstacle_center) - obstacle_radius - safety_margin
                clearances.append(dist)
        return np.array(clearances)

    # Joint limit bounds
    bounds = []
    for k in range(N_wp):
        for j in range(n_joints):
            bounds.append((ARM_JOINT_LIMITS[j, 0], ARM_JOINT_LIMITS[j, 1]))

    # Torque limit constraints (approximate: torque ~ delta_q / dt^2)
    dt_wp = 1.0 / (N_wp + 1)
    def torque_constraints(w):
        torques_ok = []
        q_prev = q_start
        for k in range(N_wp):
            q_k = w[idx(k)]
            dq = (q_k - q_prev) / dt_wp
            torques_ok.append(ARM_TORQUE_LIMIT - np.abs(dq))
            q_prev = q_k
        return np.concatenate(torques_ok)

    # Initial guess: linear interpolation
    w0 = np.zeros(n_vars)
    for k in range(N_wp):
        alpha = (k + 1) / (N_wp + 1)
        w0[idx(k)] = (1 - alpha) * q_start + alpha * q_goal

    constraints = [
        {'type': 'ineq', 'fun': obstacle_constraints},
        {'type': 'ineq', 'fun': torque_constraints},
    ]

    res = minimize(objective, w0, jac=grad_objective, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'maxiter': 300, 'ftol': 1e-10})

    # Extract trajectory
    q_traj = [q_start.copy()]
    for k in range(N_wp):
        q_traj.append(res.x[idx(k)].copy())
    q_traj.append(q_goal.copy())
    q_traj = np.array(q_traj)

    # Compute minimum clearance
    min_clear = float('inf')
    for q in q_traj:
        positions = forward_kinematics(q)
        for p in positions:
            d = norm(p - obstacle_center) - obstacle_radius
            min_clear = min(min_clear, d)

    return {
        'q_traj': q_traj,
        'success': res.success,
        'cost': res.fun,
        'min_clearance': min_clear,
        'message': res.message
    }

print("forward_kinematics, robot_arm_nlp defined.")

In [ ]:
# Solve robot arm motion planning
q_start = np.array([0.5, 0.3, -0.2])
q_goal = np.array([-0.5, 0.8, 0.4])
obstacle_center = np.array([1.0, 0.8])
obstacle_radius = 0.3

print("Solving constrained robot arm motion planning...")
arm_result = robot_arm_nlp(q_start, q_goal, obstacle_center, obstacle_radius, N_wp=12)

print(f"Success: {arm_result['success']}")
print(f"Message: {arm_result['message']}")
print(f"Smoothness cost: {arm_result['cost']:.4f}")
print(f"Min clearance from obstacle: {arm_result['min_clearance']:.4f}")
print(f"Trajectory shape: {arm_result['q_traj'].shape}")

status = "PASS" if arm_result['min_clearance'] > 0 else "FAIL"
print(f"\nrobot_arm: obstacle avoidance = {arm_result['min_clearance']:.2e} [{status}]")

# =============================================================================
# Section 9 — Visualizations
# =============================================================================

In [ ]:
# --- Visualization 1: SQP Iterates on Contour Plot (Rosenbrock + Equality) ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: SQP iterates on Rosenbrock contours
ax = axes[0]
x1 = np.linspace(-2, 2, 200)
x2 = np.linspace(-1, 3, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = (1 - X1)**2 + 100*(X2 - X1**2)**2

ax.contour(X1, X2, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis', alpha=0.7)
ax.plot(x1, 1 - x1, '--', color=SECONDARY, linewidth=2, label='$x_1 + x_2 = 1$')

x_path = np.array(result_sqp['x_hist'])
ax.plot(x_path[:, 0], x_path[:, 1], 'o-', color=PRIMARY, markersize=5, label='SQP iterates')
ax.plot(x_path[0, 0], x_path[0, 1], 's', color=ACCENT, markersize=10, label='Start')
ax.plot(x_path[-1, 0], x_path[-1, 1], '*', color=SECONDARY, markersize=15, label='Solution')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('SQP on Rosenbrock (Equality)')
ax.legend(fontsize=9)
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)

# Panel 2: KKT residual convergence
ax = axes[1]
ax.semilogy(result_sqp['kkt_hist'], 'o-', color=PRIMARY, label='SQP (equality)')
if result_sqp_ineq['kkt_hist']:
    ax.semilogy(result_sqp_ineq['kkt_hist'], 's-', color=SECONDARY, label='SQP (inequality)')
ax.axhline(KKT_TOL, color='gray', linestyle='--', alpha=0.5, label=f'tol={KKT_TOL:.0e}')
ax.set_xlabel('Iteration')
ax.set_ylabel('KKT Residual')
ax.set_title('SQP Convergence')
ax.legend(fontsize=9)

# Panel 3: SQP iterates on inequality problem
ax = axes[2]
x1 = np.linspace(-1, 4, 200)
x2 = np.linspace(-1, 4, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = (X1 - 1)**2 + (X2 - 2.5)**2

ax.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.7)

# Constraint boundaries
ax.plot(x1, (x1 + 2)/2, '--', color=SECONDARY, linewidth=1.5, label='$x_1 - 2x_2 + 2 = 0$')
ax.plot(x1, (-x1 + 6)/2, '--', color=TERTIARY, linewidth=1.5, label='$-x_1 - 2x_2 + 6 = 0$')
ax.plot(x1, (x1 + 2)/(-2) + 2, '--', color=ACCENT, linewidth=1.5)
ax.fill_between(x1, np.maximum((x1+2)/2, 0), np.minimum((-x1+6)/2, 4),
                where=((-x1+6)/2 >= (x1+2)/2), alpha=0.1, color='gray')

x_path_ineq = np.array(result_sqp_ineq['x_hist'])
ax.plot(x_path_ineq[:, 0], x_path_ineq[:, 1], 'o-', color=PRIMARY, markersize=5, label='SQP iterates')
ax.plot(x_path_ineq[-1, 0], x_path_ineq[-1, 1], '*', color=SECONDARY, markersize=15, label='Solution')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('SQP on Inequality Problem')
ax.legend(fontsize=8)
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization 2: Barrier Method Central Path + mu convergence ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Central path on contour plot
ax = axes[0]
x1 = np.linspace(-1, 4, 200)
x2 = np.linspace(-1, 4, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = (X1 - 1)**2 + (X2 - 2.5)**2

ax.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.7)

# Constraint boundaries
ax.plot(x1, (x1 + 2)/2, '--', color=SECONDARY, linewidth=1.5)
ax.plot(x1, (-x1 + 6)/2, '--', color=TERTIARY, linewidth=1.5)

bar_path = np.array(result_barrier['x_hist'])
colors_path = plt.cm.viridis(np.linspace(0, 1, len(bar_path)))
for i in range(len(bar_path)-1):
    ax.plot(bar_path[i:i+2, 0], bar_path[i:i+2, 1], '-', color=colors_path[i], linewidth=2)
ax.scatter(bar_path[:, 0], bar_path[:, 1], c=range(len(bar_path)), cmap='viridis',
           s=40, zorder=5, edgecolors='white', linewidth=0.5)
ax.plot(bar_path[-1, 0], bar_path[-1, 1], '*', color=SECONDARY, markersize=15)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Barrier Central Path')
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)

# Panel 2: mu decay
ax = axes[1]
ax.semilogy(result_barrier['mu_hist'], 'o-', color=TERTIARY, label='Barrier $\mu$')
if result_pdip['mu_hist']:
    ax.semilogy(result_pdip['mu_hist'], 's-', color=PRIMARY, label='Primal-Dual $\mu$')
ax.set_xlabel('Iteration')
ax.set_ylabel('$\mu$')
ax.set_title('Barrier Parameter Decay')
ax.legend()

# Panel 3: Primal-Dual IP path
ax = axes[2]
ax.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.7)
ax.plot(x1, (x1 + 2)/2, '--', color=SECONDARY, linewidth=1.5)
ax.plot(x1, (-x1 + 6)/2, '--', color=TERTIARY, linewidth=1.5)

pdip_path = np.array(result_pdip['x_hist'])
colors_pdip = plt.cm.viridis(np.linspace(0, 1, len(pdip_path)))
for i in range(len(pdip_path)-1):
    ax.plot(pdip_path[i:i+2, 0], pdip_path[i:i+2, 1], '-', color=colors_pdip[i], linewidth=2)
ax.scatter(pdip_path[:, 0], pdip_path[:, 1], c=range(len(pdip_path)), cmap='viridis',
           s=40, zorder=5, edgecolors='white', linewidth=0.5)
ax.plot(pdip_path[-1, 0], pdip_path[-1, 1], '*', color=SECONDARY, markersize=15)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Primal-Dual IP Path')
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization 3: Cart-Pole Swing-Up Trajectory ---

z_traj = cp_result['z_traj']
u_traj = cp_result['u_traj']
t_traj = np.arange(N_STEPS + 1) * DT
t_ctrl = np.arange(N_STEPS) * DT

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Position
axes[0, 0].plot(t_traj, z_traj[:, 0], color=PRIMARY)
axes[0, 0].set_ylabel('Cart Position $x$ [m]')
axes[0, 0].set_title('Cart Position')

# Velocity
axes[0, 1].plot(t_traj, z_traj[:, 1], color=TERTIARY)
axes[0, 1].set_ylabel('Cart Velocity $\dot{x}$ [m/s]')
axes[0, 1].set_title('Cart Velocity')

# Angle
axes[0, 2].plot(t_traj, z_traj[:, 2], color=SECONDARY)
axes[0, 2].axhline(np.pi, color='gray', linestyle='--', alpha=0.5, label='$\pi$ (upright)')
axes[0, 2].set_ylabel('Pole Angle $\theta$ [rad]')
axes[0, 2].set_title('Pole Angle')
axes[0, 2].legend()

# Angular velocity
axes[1, 0].plot(t_traj, z_traj[:, 3], color=ACCENT)
axes[1, 0].set_ylabel('Angular Vel $\dot{\theta}$ [rad/s]')
axes[1, 0].set_xlabel('Time [s]')
axes[1, 0].set_title('Angular Velocity')

# Control
axes[1, 1].step(t_ctrl, u_traj, where='post', color=PRIMARY)
axes[1, 1].axhline(15, color='red', linestyle='--', alpha=0.5)
axes[1, 1].axhline(-15, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_ylabel('Force $u$ [N]')
axes[1, 1].set_xlabel('Time [s]')
axes[1, 1].set_title('Control Input')

# Cart-pole snapshots
ax = axes[1, 2]
n_snapshots = 6
snapshot_indices = np.linspace(0, N_STEPS, n_snapshots, dtype=int)
colors_snap = plt.cm.viridis(np.linspace(0, 1, n_snapshots))

for i, (idx_s, col) in enumerate(zip(snapshot_indices, colors_snap)):
    z = z_traj[idx_s]
    cart_x = z[0]
    pole_x = cart_x + POLE_LENGTH * np.sin(z[2])
    pole_y = POLE_LENGTH * np.cos(z[2])

    # Offset each snapshot slightly for visibility
    offset = i * 0.0
    ax.plot([cart_x + offset, pole_x + offset], [0, pole_y], '-o',
            color=col, linewidth=3, markersize=6)
    ax.plot(cart_x + offset, 0, 's', color=col, markersize=10)

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Cart-Pole Snapshots')
ax.set_aspect('equal')
ax.set_ylim(-0.7, 0.7)

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization 4: Robot Arm Motion with Obstacle ---

q_traj_arm = arm_result['q_traj']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Arm configurations
ax = axes[0]
n_configs = len(q_traj_arm)
colors_arm = plt.cm.viridis(np.linspace(0, 1, n_configs))

# Draw obstacle
circle = Circle(obstacle_center, obstacle_radius, color=SECONDARY, alpha=0.4, label='Obstacle')
ax.add_patch(circle)
circle_margin = Circle(obstacle_center, obstacle_radius + 0.1,
                       color=SECONDARY, alpha=0.15, linestyle='--', fill=False)
ax.add_patch(circle_margin)

for i, (q, col) in enumerate(zip(q_traj_arm, colors_arm)):
    positions = forward_kinematics(q)
    alpha_val = 0.3 if (i > 0 and i < n_configs - 1) else 1.0
    lw = 1.5 if (i > 0 and i < n_configs - 1) else 3
    ax.plot(positions[:, 0], positions[:, 1], 'o-', color=col,
            alpha=alpha_val, linewidth=lw, markersize=4)

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Robot Arm Trajectory')
ax.set_aspect('equal')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)

# Panel 2: Joint angles over waypoints
ax = axes[1]
waypoints = np.arange(len(q_traj_arm))
ax.plot(waypoints, q_traj_arm[:, 0], 'o-', color=PRIMARY, label='$q_1$')
ax.plot(waypoints, q_traj_arm[:, 1], 's-', color=SECONDARY, label='$q_2$')
ax.plot(waypoints, q_traj_arm[:, 2], '^-', color=TERTIARY, label='$q_3$')
ax.axhline(ARM_JOINT_LIMITS[1, 0], color='gray', linestyle='--', alpha=0.3)
ax.axhline(ARM_JOINT_LIMITS[1, 1], color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Waypoint')
ax.set_ylabel('Joint Angle [rad]')
ax.set_title('Joint Trajectories')
ax.legend()

# Panel 3: Clearance from obstacle over trajectory
ax = axes[2]
clearances = []
for q in q_traj_arm:
    positions = forward_kinematics(q)
    min_d = min(norm(p - obstacle_center) - obstacle_radius for p in positions)
    clearances.append(min_d)

ax.plot(waypoints, clearances, 'o-', color=TERTIARY)
ax.axhline(0.1, color=SECONDARY, linestyle='--', label='Safety margin')
ax.axhline(0.0, color='red', linestyle='--', alpha=0.5, label='Obstacle surface')
ax.fill_between(waypoints, 0, clearances, alpha=0.2, color=TERTIARY)
ax.set_xlabel('Waypoint')
ax.set_ylabel('Min Clearance [m]')
ax.set_title('Obstacle Clearance')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Visualization 5: Method Comparison & Feasibility Convergence ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: All method paths on same contour
ax = axes[0]
x1 = np.linspace(-1, 4, 200)
x2 = np.linspace(-1, 4, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = (X1 - 1)**2 + (X2 - 2.5)**2
ax.contour(X1, X2, Z, levels=20, cmap='gray', alpha=0.4)

# Feasible region boundary
ax.plot(x1, (x1 + 2)/2, '--', color='gray', linewidth=1, alpha=0.5)
ax.plot(x1, (-x1 + 6)/2, '--', color='gray', linewidth=1, alpha=0.5)

sqp_path = np.array(result_sqp_ineq['x_hist'])
bar_path = np.array(result_barrier['x_hist'])
pdip_path = np.array(result_pdip['x_hist'])

ax.plot(sqp_path[:, 0], sqp_path[:, 1], 'o-', color=PRIMARY, label='SQP', markersize=5)
ax.plot(bar_path[:, 0], bar_path[:, 1], 's-', color=SECONDARY, label='Barrier', markersize=5)
ax.plot(pdip_path[:, 0], pdip_path[:, 1], '^-', color=TERTIARY, label='Primal-Dual IP', markersize=5)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Method Comparison: Iteration Paths')
ax.legend()
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)

# Panel 2: Constraint violation over iterations (SQP)
ax = axes[1]
violations_sqp = []
for xk in result_sqp_ineq['x_hist']:
    cv = c_ineq_test(xk)
    violations_sqp.append(np.max(np.maximum(0, cv)))

violations_bar = []
for xk in result_barrier['x_hist']:
    cv = c_ineq_test(xk)
    violations_bar.append(np.max(np.maximum(0, cv)))

ax.semilogy(range(len(violations_sqp)), [max(v, 1e-16) for v in violations_sqp],
            'o-', color=PRIMARY, label='SQP')
ax.semilogy(range(len(violations_bar)), [max(v, 1e-16) for v in violations_bar],
            's-', color=SECONDARY, label='Barrier')
ax.axhline(CONSTRAINT_TOL, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Max Constraint Violation')
ax.set_title('Feasibility Convergence')
ax.legend()

plt.tight_layout()
plt.show()

# =============================================================================
# Section 10 — Extensions & Advanced Topics
# =============================================================================

### 10.1 BFGS Hessian Approximation (Quasi-Newton SQP)

Instead of computing the exact Hessian $\nabla^2_{xx}\mathcal{L}$, use a **BFGS update**:

$$B_{k+1} = B_k - \frac{B_k s_k s_k^T B_k}{s_k^T B_k s_k} + \frac{y_k y_k^T}{y_k^T s_k}$$

where $s_k = x_{k+1} - x_k$ and $y_k = \nabla_x \mathcal{L}(x_{k+1}, \lambda_{k+1}) - \nabla_x \mathcal{L}(x_k, \lambda_{k+1})$.

This avoids second-derivative computation and maintains positive definiteness via the **damped BFGS** update (Powell's modification).

### 10.2 Filter Methods

Instead of a merit function, maintain a **filter** — a set of pairs $(h_k, f_k)$ where $h_k$ is the constraint violation. A trial point is accepted if it is not **dominated** by any filter entry:

$$\text{Accept } x^+ \text{ if } \nexists (h, f) \in \mathcal{F}: h(x^+) \ge h \text{ and } f(x^+) \ge f$$

### 10.3 Augmented Lagrangian / ADMM

The **augmented Lagrangian** combines penalty and multiplier methods:

$$\mathcal{L}_\rho(x, \lambda) = f(x) + \lambda^T c(x) + \frac{\rho}{2} \|c(x)\|^2$$

**ADMM** extends this to separable problems by splitting variables and alternating minimization.

### 10.4 Warm-Starting for Model Predictive Control (MPC)

For real-time MPC, **warm-start** the NLP solver with the shifted solution from the previous time step:

$$x_0^{(k+1)} = \text{shift}(x^{*(k)})$$

This dramatically reduces iteration count (often 1-3 SQP iterations suffice).

In [ ]:
def bfgs_sqp_solve(f, grad_f, c_eq, jac_eq, c_ineq, jac_ineq, x0,
                   max_iter=MAX_ITER, tol=KKT_TOL):
    """SQP solver with BFGS Hessian approximation (quasi-Newton).

    Uses damped BFGS to approximate the Hessian of the Lagrangian,
    avoiding second-derivative computation.

    Args:
        f: Objective function.
        grad_f: Gradient of objective. Shape: (n,).
        c_eq: Equality constraints or None.
        jac_eq: Equality Jacobian or None.
        c_ineq: Inequality constraints or None.
        jac_ineq: Inequality Jacobian or None.
        x0: Initial point. Shape: (n,).
        max_iter: Maximum iterations.
        tol: KKT tolerance.

    Returns:
        result: Dict with 'x', 'fun', 'x_hist', 'kkt_hist', 'converged'.
    """
    x = x0.copy().astype(float)
    n = len(x)
    B = np.eye(n)  # Initial BFGS approximation

    m_e = len(c_eq(x)) if c_eq is not None else 0
    m_i = len(c_ineq(x)) if c_ineq is not None else 0
    lam_eq = np.zeros(m_e)
    lam_ineq = np.zeros(m_i)
    rho = 10.0

    x_hist = [x.copy()]
    kkt_hist = []
    converged = False

    for k in range(max_iter):
        gf = grad_f(x)
        A_e = jac_eq(x) if jac_eq is not None else None
        b_e = c_eq(x) if c_eq is not None else None
        A_i = jac_ineq(x) if jac_ineq is not None else None
        b_i = c_ineq(x) if c_ineq is not None else None

        # Solve QP with BFGS Hessian
        d, lam_eq_new, lam_ineq_new = solve_qp_subproblem(B, gf, A_e, b_e, A_i, b_i)

        # KKT residual
        kkt_r = gf.copy()
        if m_e > 0:
            kkt_r += A_e.T @ lam_eq_new
        if m_i > 0:
            kkt_r += A_i.T @ lam_ineq_new
        eq_viol = norm(b_e) if b_e is not None else 0.0
        ineq_viol = norm(np.maximum(0, b_i)) if b_i is not None else 0.0
        kkt_norm = norm(kkt_r) + eq_viol + ineq_viol
        kkt_hist.append(kkt_norm)

        if kkt_norm < tol:
            converged = True
            break

        # Line search
        lam_max = max(norm(lam_eq_new, np.inf) if m_e > 0 else 0,
                      norm(lam_ineq_new, np.inf) if m_i > 0 else 0)
        rho = max(rho, 1.1 * lam_max + 0.1)

        alpha = 1.0
        for _ in range(30):
            x_trial = x + alpha * d
            phi_trial = merit_function_l1(x_trial, f, c_eq, c_ineq, rho)
            phi_curr = merit_function_l1(x, f, c_eq, c_ineq, rho)
            if phi_trial <= phi_curr - 1e-4 * alpha * norm(d)**2:
                break
            alpha *= 0.5

        x_new = x + alpha * d

        # BFGS update
        s_k = x_new - x
        # y_k = change in Lagrangian gradient
        gf_new = grad_f(x_new)
        lag_grad_new = gf_new.copy()
        lag_grad_old = gf.copy()
        if m_e > 0:
            lag_grad_new += jac_eq(x_new).T @ lam_eq_new
            lag_grad_old += A_e.T @ lam_eq_new
        if m_i > 0:
            lag_grad_new += jac_ineq(x_new).T @ lam_ineq_new
            lag_grad_old += A_i.T @ lam_ineq_new
        y_k = lag_grad_new - lag_grad_old

        # Damped BFGS (Powell's modification)
        sBs = s_k @ B @ s_k
        sy = s_k @ y_k

        if sy >= 0.2 * sBs:
            theta = 1.0
        else:
            theta = 0.8 * sBs / (sBs - sy) if abs(sBs - sy) > 1e-14 else 1.0

        r_k = theta * y_k + (1 - theta) * B @ s_k
        sr = s_k @ r_k

        if abs(sBs) > 1e-14 and abs(sr) > 1e-14:
            B = B - np.outer(B @ s_k, B @ s_k) / sBs + np.outer(r_k, r_k) / sr

        x = x_new
        lam_eq = lam_eq_new
        lam_ineq = lam_ineq_new
        x_hist.append(x.copy())

    return {
        'x': x.copy(),
        'fun': f(x),
        'x_hist': x_hist,
        'kkt_hist': kkt_hist,
        'converged': converged,
        'iterations': len(kkt_hist)
    }

# Test BFGS SQP on Rosenbrock with equality constraint
result_bfgs = bfgs_sqp_solve(
    rosenbrock, grad_rosenbrock,
    c_eq_rosen, jac_eq_rosen, None, None,
    np.array([-1.0, 2.0]), max_iter=200
)

print(f"BFGS-SQP solution: x* = [{result_bfgs['x'][0]:.6f}, {result_bfgs['x'][1]:.6f}]")
print(f"Objective: f(x*) = {result_bfgs['fun']:.6e}")
print(f"Converged: {result_bfgs['converged']}, Iterations: {result_bfgs['iterations']}")

kkt_f = result_bfgs['kkt_hist'][-1] if result_bfgs['kkt_hist'] else float('inf')
status = "PASS" if kkt_f < 1e-4 else "FAIL"
print(f"\nbfgs_sqp: KKT residual = {kkt_f:.2e} [{status}]")

In [ ]:
# --- Final Comparison: Exact vs BFGS Hessian SQP ---

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.semilogy(result_sqp['kkt_hist'], 'o-', color=PRIMARY, label='Exact Hessian SQP')
ax.semilogy(result_bfgs['kkt_hist'], 's-', color=SECONDARY, label='BFGS SQP')
ax.axhline(KKT_TOL, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('KKT Residual')
ax.set_title('Exact vs Quasi-Newton SQP')
ax.legend()

# Contour with both paths
ax = axes[1]
x1 = np.linspace(-2, 2, 200)
x2 = np.linspace(-1, 3, 200)
X1, X2 = np.meshgrid(x1, x2)
Z = (1 - X1)**2 + 100*(X2 - X1**2)**2
ax.contour(X1, X2, Z, levels=np.logspace(-1, 3.5, 20), cmap='gray', alpha=0.5)
ax.plot(x1, 1 - x1, '--', color='gray', linewidth=1, alpha=0.5)

x_exact = np.array(result_sqp['x_hist'])
x_bfgs = np.array(result_bfgs['x_hist'])

ax.plot(x_exact[:, 0], x_exact[:, 1], 'o-', color=PRIMARY, markersize=5, label='Exact Hessian')
ax.plot(x_bfgs[:, 0], x_bfgs[:, 1], 's-', color=SECONDARY, markersize=5, label='BFGS')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Iteration Paths on Rosenbrock')
ax.legend()
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Notebook complete: SQP & Interior Point Methods")
print("="*60)